In [2]:
import json

file_path = 'narrator/findings.json'

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        findings = json.load(file)
    print("JSON file loaded successfully!")
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found.")
except json.JSONDecodeError:
    print("Error: Failed to decode JSON. Please check the file format.")

JSON file loaded successfully!


In [6]:
import json
from google import genai
from google.genai import types
from google.colab import userdata

def generate_scr_narrative_offline(findings: dict) -> dict:
    """
    Fully deterministic offline fallback template generator.
    Formats Situation, Complication, and Resolution sections directly from findings
    without network calls, API keys, or external dependencies.
    """
    # Extract metrics safely with exact value fallbacks
    raw_rev = findings.get("raw_total_revenue_inr", "N/A")
    clean_rev = findings.get("cleaned_total_revenue_inr", "N/A")
    recon_delta = findings.get("duplicate_reconciliation_delta_inr", "N/A")

    returns = findings.get("return_rate_by_payment", {})
    cod_return = returns.get("COD", "N/A")
    card_return = returns.get("CARD", "N/A")
    upi_return = returns.get("UPI", "N/A")

    risk_seg = findings.get("highest_risk_segment", {})
    risk_pm = risk_seg.get("payment_method", "N/A")
    risk_tier = risk_seg.get("city_tier", "N/A")
    risk_rate = risk_seg.get("return_rate_pct", "N/A")

    peak_m = findings.get("true_peak_month", {})
    true_peak = peak_m.get("month", "N/A")
    true_peak_rev = peak_m.get("revenue_inr", "N/A")

    outlier_m = findings.get("outlier_inflated_month", {})
    outlier_month = outlier_m.get("month", "N/A")
    apparent_rev = outlier_m.get("apparent_revenue_inr", "N/A")
    corrected_rev = outlier_m.get("corrected_revenue_inr", "N/A")

    narrative = (
        f"### Situation\n"
        f"The regional operational analysis recorded a raw total revenue of INR {raw_rev}, which adjusts to a cleaned total revenue of INR {clean_rev} "
        f"following a duplicate reconciliation delta of INR {recon_delta}. True top-line peak performance occurred in {true_peak} with revenue of INR {true_peak_rev}.\n\n"
        f"### Complication\n"
        f"Revenue reporting in {outlier_month} was distorted by an outlier inflation showing an apparent revenue of INR {apparent_rev} against a corrected revenue of INR {corrected_rev}. "
        f"Furthermore, return rates across payment methods show significant divergence with COD at {cod_return}%, UPI at {upi_return}%, and CARD at {card_return}%. "
        f"The highest risk segment is concentrated in {risk_pm} across Tier {risk_tier} cities, experiencing a return rate of {risk_rate}%.\n\n"
        f"### Resolution\n"
        f"Regional operations and finance heads must adjust historical baseline forecasting to reflect the corrected {outlier_month} revenue figures "
        f"and institute tighter verification protocols for {risk_pm} orders in Tier {risk_tier} markets to mitigate return losses."
    )

    return {
        "status": "success_offline",
        "narrative": narrative,
        "tokens": 0,
        "finish_reason": "OFFLINE_TEMPLATE"
    }


def generate_scr_narrative(findings: dict) -> dict:
    """
    Generates an SCR narrative via Gemini API when available.
    Automatically falls back to generate_scr_narrative_offline on missing API keys or errors.
    """
    api_key = None
    try:
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

    if not api_key:
        return generate_scr_narrative_offline(findings)

    try:
        client = genai.Client(
            api_key=api_key,
            http_options=types.HttpOptions(timeout=13000)
        )

        system_instruction = (
            "You are a senior data analyst writing for Mamaearth's regional ops and finance heads.\n\n"
            "Formatting & Data Rules:\n"
            "1. Structure: You must structure your response into exactly three labeled sections: Situation, Complication, and Resolution.\n"
            "2. Data Integrity: Every number, metric, or statistic in your output must come directly from the supplied data and appear with the exact same value. Do not invent, extrapolate, or estimate any numbers."
        )

        prompt = f"Write the narrative based on these exact findings:\n\n{findings}"

        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt,
            config={
                'system_instruction': system_instruction,
                'temperature': 0.0,
                'max_output_tokens': 1500
            }
        )

        usage = getattr(response, 'usage_metadata', None)
        token_count = getattr(usage, 'total_token_count', None) if usage else None

        return {
            "status": "success",
            "narrative": response.text,
            "tokens": token_count,
        }

    except Exception:
        return generate_scr_narrative_offline(findings)

In [7]:
output = generate_scr_narrative(findings)
output

{'status': 'success_offline',
 'narrative': '### Situation\nThe regional operational analysis recorded a raw total revenue of INR 99860.2, which adjusts to a cleaned total revenue of INR 97358.3 following a duplicate reconciliation delta of INR 2501.9. True top-line peak performance occurred in 2026-03 with revenue of INR 20318.9.\n\n### Complication\nRevenue reporting in 2026-01 was distorted by an outlier inflation showing an apparent revenue of INR 29582.1 against a corrected revenue of INR 11637.1. Furthermore, return rates across payment methods show significant divergence with COD at 44.4%, UPI at 18.9%, and CARD at 14.7%. The highest risk segment is concentrated in COD across Tier 2 cities, experiencing a return rate of 54.5%.\n\n### Resolution\nRegional operations and finance heads must adjust historical baseline forecasting to reflect the corrected 2026-01 revenue figures and institute tighter verification protocols for COD orders in Tier 2 markets to mitigate return losses.',

In [8]:
import os
import re

def verify_narrative(narrative_text: str) -> bool:
    """
    Asserts all 5 required figures are present in the narrative text.
    Prints a PASS/FAIL line per figure.
    """
    # Strip commas so '97,358.30' and '97358.3' match uniformly
    normalized_text = narrative_text.replace(',', '')

    checks = [
        ("Cleaned Total Revenue (97358.3)", r"97358\.30?"),
        ("COD Return Rate (44.4)", r"44\.4"),
        ("Highest-Risk Segment Return Rate (54.5)", r"54\.5"),
        ("Reconciliation Delta (2501.9)", r"2501\.90?"),
        ("True Peak Month & Revenue (March/2026-03 and 20318.9)", r"(March|2026-03).{0,120}20318\.90?|20318\.90?.{0,120}(March|2026-03)"),
    ]

    all_passed = True
    print("\n--- Numerical Verification Results ---")
    for label, pattern in checks:
        if re.search(pattern, normalized_text, re.IGNORECASE):
            print(f"[PASS] {label}")
        else:
            print(f"[FAIL] {label}")
            all_passed = False

    print("---------------------------------------")
    print(f"Overall Checker Result: {'PASSED' if all_passed else 'FAILED'}\n")
    return all_passed


# 1. Pass your existing `findings` variable into the function
result = generate_scr_narrative(findings)

# 2. Save narrative to file for grader evaluation
os.makedirs("narrator", exist_ok=True)
sample_output_path = os.path.join("narrator", "sample_output.txt")

with open(sample_output_path, "w", encoding="utf-8") as f:
    f.write(result["narrative"])

print(f"Narrative successfully saved to '{sample_output_path}'")

# 3. Read saved output and verify the 5 figures
with open(sample_output_path, "r", encoding="utf-8") as f:
    saved_text = f.read()

verify_narrative(saved_text)

Narrative successfully saved to 'narrator/sample_output.txt'

--- Numerical Verification Results ---
[PASS] Cleaned Total Revenue (97358.3)
[PASS] COD Return Rate (44.4)
[PASS] Highest-Risk Segment Return Rate (54.5)
[PASS] Reconciliation Delta (2501.9)
[PASS] True Peak Month & Revenue (March/2026-03 and 20318.9)
---------------------------------------
Overall Checker Result: PASSED



True